In [28]:
import re
from collections import defaultdict
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc

In [8]:
data_dir = Path("../data/")
[i for i in data_dir.glob("GSM825*")]

[PosixPath('../data/GSM8257571_Untreated1_barcodes.tsv.gz'),
 PosixPath('../data/GSM8257571_Untreated1_matrix.mtx.gz'),
 PosixPath('../data/GSM8257565_7day1_gene_panel.json.gz'),
 PosixPath('../data/GSM8257565_7day1_barcodes.tsv.gz'),
 PosixPath('../data/GSM8257565_7day1_transcripts.csv.gz'),
 PosixPath('../data/GSM8257565_7day1_matrix.mtx.gz'),
 PosixPath('../data/GSM8257565_7day1_features.tsv.gz'),
 PosixPath('../data/GSM8257571_Untreated1_cells.csv.gz'),
 PosixPath('../data/GSM8257571_Untreated1_features.tsv.gz'),
 PosixPath('../data/GSM8257571_Untreated1_gene_panel.json.gz'),
 PosixPath('../data/GSM8257571_Untreated1_transcripts.csv.gz'),
 PosixPath('../data/GSM8257565_7day1_cells.csv.gz')]

In [26]:
pattern = re.compile(r'^(?P<prefix>.*)cells\.csv(\.gz)?$')


lookup = defaultdict(dict)
for filename in data_dir.glob("GSM825*"):
    match = pattern.match(filename.as_posix())
    if match is None:
        continue
    prefix = match.group("prefix")
    sample = prefix.split("_")[1]
    lookup[sample] = prefix

lookup

defaultdict(dict,
            {'Untreated1': '../data/GSM8257571_Untreated1_',
             '7day1': '../data/GSM8257565_7day1_'})

In [33]:
def read_xenium(sample, prefix):
    """Reads 10x Xenium formatted data into AnnData object."""
    adata = sc.read_10x_mtx(data_dir, prefix=prefix)
    df = pd.read_csv(prefix + "cells.csv.gz")
    df.set_index(adata.obs_names, inplace=True)

    # Populating spatial attribute of AnnData.obs
    adata.obs = df.copy()
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()
    adata.obs["sample"] = sample

    # Basic filtering
    sc.pp.filter_cells(adata, min_genes=10)
    sc.pp.filter_genes(adata, min_cells=5)
    return adata

adata = ad.concat(
    [read_xenium(sample, prefix) for (sample, prefix) in lookup.items()],
    index_unique="_"  # or None if you don't want uniqueness enforced
)

In [34]:
adata

AnnData object with n_obs × n_vars = 202487 × 347
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'sample', 'n_genes'
    obsm: 'spatial'